<a href="https://colab.research.google.com/github/hcristosm/image_batch_upscale/blob/main/batch_upscale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 📸 Real-ESRGAN & GFPGAN Automated Batch Upscaler
# ==============================================================================
# Instructions for GitHub Users:
# 1. Ensure GPU acceleration is active: Runtime > Change runtime type > T4 GPU
# 2. Run this cell by clicking the Play (▶) button or pressing Ctrl + Enter
# 3. Choose your image(s) or a single .zip file when prompted
# 4. Your 4x upscaled images will automatically download as 'upscaled_photos.zip'
# ==============================================================================

import os
import shutil
import zipfile
from PIL import Image
from google.colab import files
from IPython.display import clear_output

# ------------------------------------------------------------------------------
# STEP 1: Environment Setup & Dependencies Installation
# ------------------------------------------------------------------------------
# Check if the repository is already cloned to avoid re-installing on repeated runs
if not os.path.exists('/content/Real-ESRGAN'):
    print("⏳ Setting up environment and downloading AI weights (first run only)...")

    # 1.1 Clone the official Real-ESRGAN repository
    !git clone https://github.com/xinntao/Real-ESRGAN.git 2> /dev/null
    os.chdir('/content/Real-ESRGAN')

    # 1.2 Install required libraries silently
    !pip install -q basicsr facexlib gfpgan
    !pip install -q -r requirements.txt
    !python setup.py develop > /dev/null

    # 1.3 Download pre-trained model weights (RealESRGAN_x4plus for real-world photos)
    !wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights

    # 1.4 Apply compatibility patch for PyTorch / Torchvision in Python 3.12+ runtime
    !sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
else:
    # Set current working directory if already installed
    os.chdir('/content/Real-ESRGAN')

clear_output()
print("✅ Environment ready! All dependencies and compatibility patches applied.")

# ------------------------------------------------------------------------------
# STEP 2: Workspace Initialization
# ------------------------------------------------------------------------------
# Reset working folders to ensure images from previous sessions are not mixed
input_dir = 'inputs'
output_dir = 'results'

if os.path.exists(input_dir):
    shutil.rmtree(input_dir)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# STEP 3: File Upload & Pre-processing
# ------------------------------------------------------------------------------
print("\n📦 STEP 1/4: Select your photo(s) or a .zip archive...")
uploaded = files.upload()

if uploaded:
    print("\n⚙️ STEP 2/4: Preparing files for AI inference...")

    # 3.1 Handle zip files (extracts all images into a flat folder structure)
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    # Skip subdirectories to unpack everything straight into input_dir
                    if not member.endswith('/'):
                        source = zip_ref.open(member)
                        target_name = os.path.basename(member)
                        if target_name:
                            with open(os.path.join(input_dir, target_name), "wb") as target:
                                shutil.copyfileobj(source, target)
            os.remove(filename) # Clean up original upload
        else:
            # Move individual images to input directory
            shutil.move(filename, os.path.join(input_dir, filename))

    # 3.2 Ensure 3-channel RGB format
    # Note: GFPGAN face restoration fails on 1-channel Grayscale images.
    # Converting all images to RGB prevents runtime crashes without altering visual appearance.
    file_list = os.listdir(input_dir)
    print(f"📸 Found {len(file_list)} image(s). Validating color formats...")
    for file in file_list:
        img_path = os.path.join(input_dir, file)
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img.convert('RGB').save(img_path)
        except Exception:
            pass # Skip non-image files if any

    # --------------------------------------------------------------------------
    # STEP 4: AI Super-Resolution Execution
    # --------------------------------------------------------------------------
    # Parameter Breakdown:
    # -n RealESRGAN_x4plus : Uses the standard 4x scale model for general photos.
    # -i inputs -o results  : Defines input and output directories.
    # --outscale 4          : Upscales resolution by a factor of 4.
    # --face_enhance        : Enables GFPGAN to reconstruct facial features.
    # --tile 512            : Splits processing into chunks to prevent GPU memory allocation errors (OOM).
    print("\n🚀 STEP 3/4: Processing AI 4x Upscale...")
    print("💡 Customization: Remove '--face_enhance' for non-portrait photos to increase speed.\n")

    !python inference_realesrgan.py -n RealESRGAN_x4plus -i inputs -o results --outscale 4 --face_enhance --tile 512

    # --------------------------------------------------------------------------
    # STEP 5: Output Verification & Download
    # --------------------------------------------------------------------------
    results = os.listdir(output_dir)
    if len(results) == 0:
        print("\n❌ Processing completed, but no output files were generated.")
        print("Please check the terminal output above to troubleshoot errors.")
    else:
        print(f"\n✅ STEP 4/4: Success! Processed {len(results)} image(s).")
        print("🗜️ Zipping output directory and initiating download...")

        # Package processed results
        shutil.make_archive('upscaled_photos', 'zip', 'results')

        # Download ZIP file automatically in browser
        files.download('upscaled_photos.zip')
else:
    print("\n❌ No files uploaded. Re-run this cell to try again.")